# Football-LLM: Evaluation Harness

Compare **base model** vs **fine-tuned model** vs **baselines** on the 2022 World Cup held-out eval set.

## Metrics
- **Result Accuracy**: Correct prediction of home_win / draw / away_win
- **Score Exact Match**: Correct exact score
- **Score MAE**: Mean absolute error on home_goals and away_goals
- **Parse Rate**: % of model outputs that could be parsed into a valid prediction
- **Memorization Test**: Named vs anonymized accuracy to detect knowledge leakage

## Models Evaluated
1. **Base Llama 3.1 8B Instruct** (no fine-tuning)
2. **Football-LLM** (QLoRA fine-tuned adapter)
3. **Baseline: Always Home Win** (most common result in WC history)
4. **Baseline: Random** (weighted by train set distribution)

## 1. Setup

In [ ]:
%pip install -q transformers accelerate bitsandbytes peft datasets

from huggingface_hub import login
login(token="")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00


In [2]:
import torch
import json
import re
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load Eval Data

In [3]:
!git clone https://github.com/zanwenfu/football-llm.git 2>/dev/null || true

eval_dataset = load_dataset("json", data_files="football-llm/data/training/eval.jsonl", split="train")
print(f"Eval samples: {len(eval_dataset)}")

# Separate named vs anonymized samples for memorization test
named_samples = []
anon_samples = []
for sample in eval_dataset:
    user_msg = sample['messages'][1]['content']
    if 'Team A' in user_msg and 'Team B' in user_msg:
        anon_samples.append(sample)
    else:
        named_samples.append(sample)

print(f"Named team samples: {len(named_samples)}")
print(f"Anonymized samples: {len(anon_samples)}")

Generating train split: 0 examples [00:00, ? examples/s]

Eval samples: 128
Named team samples: 64
Anonymized samples: 64


## 3. Prediction Parser

Robust parser that extracts prediction and score from both model output and ground truth.

In [4]:
def parse_prediction(text):
    """Extract match result and score from model output text.

    Returns dict with:
      - result: 'home_win' | 'away_win' | 'draw' | None
      - home_goals: int | None
      - away_goals: int | None
      - parsed: bool
    """
    result = None
    home_goals = None
    away_goals = None

    text_lower = text.lower()

    # --- Parse result ---
    # Pattern 1: "Prediction: X win" or "Prediction: draw"
    pred_match = re.search(r'prediction[:\s]+(.*?)(?:\n|$)', text_lower)
    if pred_match:
        pred_text = pred_match.group(1).strip()
        if 'draw' in pred_text:
            result = 'draw'
        elif 'home' in pred_text or 'team a' in pred_text:
            result = 'home_win'
        elif 'away' in pred_text or 'team b' in pred_text:
            result = 'away_win'
        else:
            # Named team — check if it says "X win"
            # We'll need to match against the teams in context
            if 'win' in pred_text:
                result = 'has_winner'  # will resolve later

    # --- Parse score ---
    # Pattern: "Score: X N - M Y" or "N - M" or "N-M"
    score_patterns = [
        r'score[:\s]+.*?(\d+)\s*[-–]\s*(\d+)',  # "Score: Team A 2 - 1 Team B"
        r'(\d+)\s*[-–]\s*(\d+)',                  # bare "2 - 1"
    ]
    for pattern in score_patterns:
        match = re.search(pattern, text)
        if match:
            home_goals = int(match.group(1))
            away_goals = int(match.group(2))
            break

    # ALWAYS derive result from score when available.
    # Score is more precise than text label — the model sometimes
    # outputs contradictory labels (e.g., "home_win" with score 0-1).
    if home_goals is not None and away_goals is not None:
        if home_goals > away_goals:
            result = 'home_win'
        elif home_goals < away_goals:
            result = 'away_win'
        else:
            result = 'draw'

    parsed = result is not None
    return {
        'result': result,
        'home_goals': home_goals,
        'away_goals': away_goals,
        'parsed': parsed,
    }


def parse_ground_truth(assistant_msg):
    """Parse ground truth from the assistant message in eval data."""
    return parse_prediction(assistant_msg)


# Test parser on ground truth samples
for i in range(3):
    gt_text = eval_dataset[i]['messages'][2]['content']
    parsed = parse_ground_truth(gt_text)
    print(f"Sample {i}: {parsed}")
    print(f"  Text: {gt_text[:120]}...")
    print()

Sample 0: {'result': 'away_win', 'home_goals': 0, 'away_goals': 2, 'parsed': True}
  Text: Prediction: away_win
Score: 0-2
Reasoning: Qatar's squad has higher goal output (199 vs 130). Qatar's top scorer is more...

Sample 1: {'result': 'away_win', 'home_goals': 0, 'away_goals': 2, 'parsed': True}
  Text: Prediction: away_win
Score: 0-2
Reasoning: Team A's squad has higher goal output (199 vs 130). Team A's top scorer is mo...

Sample 2: {'result': 'home_win', 'home_goals': 6, 'away_goals': 2, 'parsed': True}
  Text: Prediction: home_win
Score: 6-2
Reasoning: England's squad has higher goal output (308 vs 139). England has a higher ave...



## 4. Load Models

In [5]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_ID = "zanwenfu/football-llm-qlora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer from adapter (has the correct pad token config)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
print(f"Base model loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Base model loaded. GPU: 5.7 GB


## 5. Run Inference — Batch Evaluation Function

In [6]:
def run_eval(model, tokenizer, samples, model_name="model", max_new_tokens=300):
    """Run inference on all samples and return parsed predictions."""
    results = []
    model.eval()

    for i, sample in enumerate(samples):
        messages = sample['messages'][:2]  # system + user
        gt_text = sample['messages'][2]['content']
        gt = parse_ground_truth(gt_text)

        # Generate
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            input_text, return_tensors="pt", truncation=True, max_length=768
        ).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,       # low temp for deterministic eval
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1, # reduce repetitive output
            )

        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        pred = parse_prediction(response)

        results.append({
            'sample_idx': i,
            'gt': gt,
            'pred': pred,
            'raw_output': response[:500],
        })

        if (i + 1) % 10 == 0:
            print(f"  [{model_name}] {i+1}/{len(samples)} done")

    return results


def compute_metrics(results):
    """Compute eval metrics from a list of results."""
    total = len(results)
    parsed = sum(1 for r in results if r['pred']['parsed'])

    # Only compute accuracy on parsed predictions
    correct_result = 0
    correct_score = 0
    home_goal_errors = []
    away_goal_errors = []

    for r in results:
        if not r['pred']['parsed'] or not r['gt']['parsed']:
            continue

        if r['pred']['result'] == r['gt']['result']:
            correct_result += 1

        if (r['pred']['home_goals'] == r['gt']['home_goals'] and
            r['pred']['away_goals'] == r['gt']['away_goals']):
            correct_score += 1

        if r['pred']['home_goals'] is not None and r['gt']['home_goals'] is not None:
            home_goal_errors.append(abs(r['pred']['home_goals'] - r['gt']['home_goals']))
            away_goal_errors.append(abs(r['pred']['away_goals'] - r['gt']['away_goals']))

    parse_rate = parsed / total if total > 0 else 0
    result_acc = correct_result / parsed if parsed > 0 else 0
    score_acc = correct_score / parsed if parsed > 0 else 0
    goal_mae = np.mean(home_goal_errors + away_goal_errors) if home_goal_errors else float('nan')

    return {
        'total': total,
        'parsed': parsed,
        'parse_rate': parse_rate,
        'result_accuracy': result_acc,
        'score_exact_match': score_acc,
        'goal_mae': goal_mae,
    }

print("Eval functions defined.")

Eval functions defined.


## 6. Evaluate Base Model (no fine-tuning)

In [7]:
# Base model results from previous run (no need to re-evaluate)
# Base Llama 3.1 8B Instruct on 2022 WC eval set (128 samples)
base_metrics = {
    'total': 128,
    'parsed': 128,
    'parse_rate': 1.000,
    'result_accuracy': 0.453,
    'score_exact_match': 0.000,
    'goal_mae': 2.133,
}
base_results = None  # raw results not available (from previous session)

print("=== BASE MODEL (hardcoded from previous run) ===")
for k, v in base_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")


=== BASE MODEL (hardcoded from previous run) ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.453
  score_exact_match: 0.000
  goal_mae: 2.133


## 7. Evaluate Fine-tuned Model

In [8]:
# Load adapter on top of base model
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
ft_model.eval()
print(f"Adapter loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("\nEvaluating FINE-TUNED model on all eval samples...")
ft_results = run_eval(ft_model, tokenizer, eval_dataset, model_name="fine-tuned")
ft_metrics = compute_metrics(ft_results)

print(f"\n=== FINE-TUNED MODEL ===")
for k, v in ft_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Adapter loaded. GPU: 5.9 GB

Evaluating FINE-TUNED model on all eval samples...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

  [fine-tuned] 110/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  [fine-tuned] 120/128 done


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



=== FINE-TUNED MODEL ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.523
  score_exact_match: 0.297
  goal_mae: 1.285


In [9]:
# Re-parse ft_results with fixed parser (score overrides text label)
reparse_count = 0
for r, sample in zip(ft_results, eval_dataset):
    old_result = r['pred']['result']
    r['pred'] = parse_prediction(r['raw_output'])
    r['gt'] = parse_ground_truth(sample['messages'][2]['content'])
    if old_result != r['pred']['result']:
        reparse_count += 1

print(f"Re-parsed {reparse_count} predictions where score overrode text label")

ft_metrics = compute_metrics(ft_results)
print(f"\n=== FINE-TUNED MODEL (fixed parser) ===")
for k, v in ft_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

Re-parsed 0 predictions where score overrode text label

=== FINE-TUNED MODEL (fixed parser) ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.523
  score_exact_match: 0.297
  goal_mae: 1.285


## 8. Baselines

In [10]:
def run_baseline(samples, strategy='always_home'):
    """Run baseline predictions."""
    np.random.seed(42)
    results = []

    # WC distribution from training data (2010+2014+2018): ~40% home, ~35% away, ~25% draw
    wc_dist = {'home_win': 0.40, 'away_win': 0.35, 'draw': 0.25}

    for i, sample in enumerate(samples):
        gt = parse_ground_truth(sample['messages'][2]['content'])

        if strategy == 'always_home':
            pred_result = 'home_win'
            pred_home, pred_away = 1, 0
        elif strategy == 'random_weighted':
            pred_result = np.random.choice(
                list(wc_dist.keys()), p=list(wc_dist.values())
            )
            if pred_result == 'home_win':
                pred_home, pred_away = np.random.choice([1,2]), 0
            elif pred_result == 'away_win':
                pred_home, pred_away = 0, np.random.choice([1,2])
            else:
                goals = np.random.choice([0,1])
                pred_home, pred_away = goals, goals
        elif strategy == 'always_draw':
            pred_result = 'draw'
            pred_home, pred_away = 1, 1

        results.append({
            'sample_idx': i,
            'gt': gt,
            'pred': {
                'result': pred_result,
                'home_goals': pred_home,
                'away_goals': pred_away,
                'parsed': True,
            },
            'raw_output': f'{strategy}: {pred_home}-{pred_away}',
        })

    return results

# Run baselines
home_results = run_baseline(eval_dataset, 'always_home')
random_results = run_baseline(eval_dataset, 'random_weighted')
draw_results = run_baseline(eval_dataset, 'always_draw')

home_metrics = compute_metrics(home_results)
random_metrics = compute_metrics(random_results)
draw_metrics = compute_metrics(draw_results)

print("=== ALWAYS HOME WIN ===")
for k, v in home_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== RANDOM WEIGHTED ===")
for k, v in random_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== ALWAYS DRAW ===")
for k, v in draw_metrics.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

=== ALWAYS HOME WIN ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.453
  score_exact_match: 0.109
  goal_mae: 1.109

=== RANDOM WEIGHTED ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.359
  score_exact_match: 0.078
  goal_mae: 1.273

=== ALWAYS DRAW ===
  total: 128
  parsed: 128
  parse_rate: 1.000
  result_accuracy: 0.234
  score_exact_match: 0.078
  goal_mae: 0.969


## 9. Memorization Test (Named vs Anonymized)

In [11]:
# Split results by named vs anonymized
if named_samples and anon_samples:
    ft_named = [r for r, s in zip(ft_results, eval_dataset)
                if 'Team A' not in s['messages'][1]['content']]
    ft_anon = [r for r, s in zip(ft_results, eval_dataset)
               if 'Team A' in s['messages'][1]['content']]

    print("=== MEMORIZATION TEST ===")
    print(f"\nFine-tuned model:")
    if ft_named:
        m = compute_metrics(ft_named)
        print(f"  Named teams   — result_acc: {m['result_accuracy']:.3f}, parse_rate: {m['parse_rate']:.3f} (n={m['total']})")
    if ft_anon:
        m = compute_metrics(ft_anon)
        print(f"  Anonymized    — result_acc: {m['result_accuracy']:.3f}, parse_rate: {m['parse_rate']:.3f} (n={m['total']})")

    print(f"\nBase model (from previous session):")
    print(f"  Overall       — result_acc: 0.453, parse_rate: 1.000 (n=128)")
    print(f"  (Named vs anon breakdown not available — base model raw results not saved)")

    print("\nInterpretation:")
    print("  If fine-tuned model is similar on named vs anon → learned from stats, not names")
    print("  If much better on named → still relying on memorization")
else:
    print("Could not split named vs anonymized samples.")


=== MEMORIZATION TEST ===

Fine-tuned model:
  Named teams   — result_acc: 0.500, parse_rate: 1.000 (n=64)
  Anonymized    — result_acc: 0.547, parse_rate: 1.000 (n=64)

Base model (from previous session):
  Overall       — result_acc: 0.453, parse_rate: 1.000 (n=128)
  (Named vs anon breakdown not available — base model raw results not saved)

Interpretation:
  If fine-tuned model is similar on named vs anon → learned from stats, not names
  If much better on named → still relying on memorization


## 10. Results Summary

In [12]:
print("=" * 75)
print(f"{'Model':<25} {'Parse%':>8} {'Result Acc':>12} {'Score EM':>10} {'Goal MAE':>10}")
print("=" * 75)

all_metrics = [
    ('Base Llama 3.1 8B', base_metrics),
    ('Football-LLM (QLoRA)', ft_metrics),
    ('Baseline: Always Home', home_metrics),
    ('Baseline: Random', random_metrics),
    ('Baseline: Always Draw', draw_metrics),
]

for name, m in all_metrics:
    parse_pct = f"{m['parse_rate']*100:.1f}%"
    result_acc = f"{m['result_accuracy']*100:.1f}%"
    score_em = f"{m['score_exact_match']*100:.1f}%"
    goal_mae = f"{m['goal_mae']:.2f}" if not np.isnan(m['goal_mae']) else 'N/A'
    print(f"{name:<25} {parse_pct:>8} {result_acc:>12} {score_em:>10} {goal_mae:>10}")

print("=" * 75)
print(f"\nEval set: 2022 FIFA World Cup ({len(eval_dataset)} samples)")
print(f"Train set: 2010 + 2014 + 2018 World Cups")


Model                       Parse%   Result Acc   Score EM   Goal MAE
Base Llama 3.1 8B           100.0%        45.3%       0.0%       2.13
Football-LLM (QLoRA)        100.0%        52.3%      29.7%       1.29
Baseline: Always Home       100.0%        45.3%      10.9%       1.11
Baseline: Random            100.0%        35.9%       7.8%       1.27
Baseline: Always Draw       100.0%        23.4%       7.8%       0.97

Eval set: 2022 FIFA World Cup (128 samples)
Train set: 2010 + 2014 + 2018 World Cups


## 11. Error Analysis — Sample Predictions

In [13]:
# Show 5 examples where fine-tuned model got it right or wrong
print("=== FINE-TUNED MODEL: Sample Predictions ===")
correct = [r for r in ft_results if r['pred']['parsed'] and r['gt']['parsed']
           and r['pred']['result'] == r['gt']['result']]
wrong = [r for r in ft_results if r['pred']['parsed'] and r['gt']['parsed']
         and r['pred']['result'] != r['gt']['result']]
unparsed = [r for r in ft_results if not r['pred']['parsed']]

print(f"\nCorrect: {len(correct)}, Wrong: {len(wrong)}, Unparsed: {len(unparsed)}")

print("\n--- CORRECT predictions (first 3) ---")
for r in correct[:3]:
    print(f"  GT: {r['gt']['result']} ({r['gt']['home_goals']}-{r['gt']['away_goals']})")
    print(f"  Pred: {r['pred']['result']} ({r['pred']['home_goals']}-{r['pred']['away_goals']})")
    print(f"  Output: {r['raw_output'][:150]}...")
    print()

print("--- WRONG predictions (first 3) ---")
for r in wrong[:3]:
    print(f"  GT: {r['gt']['result']} ({r['gt']['home_goals']}-{r['gt']['away_goals']})")
    print(f"  Pred: {r['pred']['result']} ({r['pred']['home_goals']}-{r['pred']['away_goals']})")
    print(f"  Output: {r['raw_output'][:150]}...")
    print()

print("--- UNPARSED outputs (first 3) ---")
for r in unparsed[:3]:
    print(f"  GT: {r['gt']['result']}")
    print(f"  Output: {r['raw_output'][:200]}...")
    print()

=== FINE-TUNED MODEL: Sample Predictions ===

Correct: 67, Wrong: 61, Unparsed: 0

--- CORRECT predictions (first 3) ---
  GT: away_win (0-2)
  Pred: away_win (0-1)
  Output: Prediction: home_win
Score: 0-1
Reasoning: Team A's squad has higher goal output (200 vs 131). Team A's top scorer is more prolific (59 goals vs 40). ...

  GT: home_win (6-2)
  Pred: home_win (6-2)
  Output: Prediction: home_win
Score: 6-2
Reasoning: England's squad has higher goal output (309 vs 141). England's top scorer is more prolific (110 vs 78 goals...

  GT: away_win (0-2)
  Pred: away_win (0-2)
  Output: Prediction: home_win
Score: 0-2
Reasoning: Netherlands's squad has higher goal output (213 vs 116). Nederland's top scorer is more prolific (59 vs 32 ...

--- WRONG predictions (first 3) ---
  GT: away_win (0-2)
  Pred: home_win (2-0)
  Output: Prediction: home_win
Score: 2-0
Reasoning: Ecuador's squad has higher goal output (190 vs 149). Ecuador's top scorer is more prolific (40 vs 10 goals)...

  GT: h

## 12. Save Results

In [14]:
# Save full results for the repo
eval_output = {
    'eval_set': '2022 FIFA World Cup',
    'train_set': '2010 + 2014 + 2018 World Cups',
    'num_eval_samples': len(eval_dataset),
    'metrics': {
        'base_model': base_metrics,
        'fine_tuned': ft_metrics,
        'baseline_always_home': home_metrics,
        'baseline_random': random_metrics,
        'baseline_always_draw': draw_metrics,
    },
}

with open('eval_results.json', 'w') as f:
    json.dump(eval_output, f, indent=2, default=str)

print("Results saved to eval_results.json")
print("Download it and add to your repo: football-llm/results/eval_results.json")

# Download
try:
    from google.colab import files
    files.download('eval_results.json')
except:
    pass


Results saved to eval_results.json
Download it and add to your repo: football-llm/results/eval_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>